# Time Series Analysis

In this notebook, we explore the data understanding and preparation steps for ECG time series collected from multiple subjects. The workflow includes loading and organizing the raw ECG data, visualizing and cleaning the signals, applying various noise reduction techniques, and extracting relevant features for further analysis. We also demonstrate methods for detecting peaks, assessing data quality, and preparing the dataset for downstream tasks such as classification or clustering. This comprehensive approach ensures that the ECG signals are ready for robust and meaningful time series analysis.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from timeit import default_timer as timer
from sklearn.preprocessing import StandardScaler

In [ ]:
FOLDER_PATH = Path("../../data/time-series")  

subdirs = sorted([d for d in FOLDER_PATH.iterdir() if d.is_dir()])

print(f"Total directories found: {len(subdirs)}\n")

subject_infos = []

# Extract subject_id and hadm_id (record) from directory names
for subdir in subdirs:
    # Split directory name: format is "subjectid_hadmid"
    parts = subdir.name.split('_')
    if len(parts) >= 2:
        subject_id = parts[0]
        hadm_id = parts[1]  # This is the record/admission ID
        subject_infos.append({
            'subject_id': subject_id,
            'record_name': hadm_id,
            'folder_name': subdir.name
        })

# Convert to DataFrame for easier manipulation
subject_infos_df = pd.DataFrame(subject_infos)

subject_infos_df.shape

In [ ]:
timeseries_dict = {}

for record in subject_infos_df['record_name'].unique():
    record_path = FOLDER_PATH / f"{subject_infos_df[subject_infos_df['record_name'] == record]['folder_name'].values[0]}" / record
    try:
        # Read the entire signal
        signal, fields = wfdb.rdsamp(str(record_path))
        fs = fields['fs']  # Sampling frequency
        
        # Use the actual length of the signal
        num_samples = signal.shape[0]
        num_channels = signal.shape[1]
        duration_seconds = num_samples / fs
        
        # Create time axis for the entire signal
        time_axis = np.arange(num_samples) / fs
        
        # Create a unique time series concatenating all channels sequentially
        # Flatten column-wise: Ch1 samples, then Ch2 samples, then Ch3 samples, etc.
        concatenated_signal = signal.flatten(order='F')  # 'F' for Fortran-style (column-major)
        
        total_length = concatenated_signal.shape[0]
        
        # Calculate start time (in seconds) for each channel in the concatenated signal
        signal_start_times = []
        for i in range(10):
            start_sample = i * num_samples
            start_time = start_sample / fs
            signal_start_times.append({
                'channel_index': i,
                'channel_name': fields['sig_name'][i],
                'start_sample': start_sample,
                'start_time_seconds': start_time,
                'end_sample': start_sample + num_samples - 1,
                'end_time_seconds': (start_sample + num_samples - 1) / fs
            })
        
        # Store in dictionary
        timeseries_dict[record] = {
            'signal': concatenated_signal,
            'fs': fs,
            'num_channels': num_channels,
            'samples_per_channel': num_samples,
            'total_length': total_length,
            'duration_seconds': duration_seconds,
            'signal_names': fields['sig_name'],
            'signal_start_times': signal_start_times
        }

    except Exception as e:
        print(f"Could not read record {record}: {e}")

print(f"\nTotal records processed: {len(timeseries_dict)}")

# Print the summary of the first record
first_record = list(timeseries_dict.keys())[0]
print(f"\nSummary for record {first_record}:")
record_info = timeseries_dict[first_record]
print(f"  Sampling Frequency: {record_info['fs']} Hz")
print(f"  Number of Channels: {record_info['num_channels']}")
print(f"  Samples per Channel: {record_info['samples_per_channel']}")
print(f"  Total Length of Concatenated Signal: {record_info['total_length']} samples")
print(f"  Duration: {record_info['duration_seconds']:.2f} seconds")
print(f"  Signal Names: {record_info['signal_names']}")
print("  Signal Start Times per Channel:")
for sig_time in record_info['signal_start_times']:
    print(f"    Channel {sig_time['channel_index']} ({sig_time['channel_name']}): "
          f"Samples {sig_time['start_sample']} to {sig_time['end_sample']} "
          f"({sig_time['start_time_seconds']:.2f}s to {sig_time['end_time_seconds']:.2f}s)")

In [ ]:
# Print the frequency of sampling for the first record
first_record = list(timeseries_dict.keys())[0]
fs_first_record = timeseries_dict[first_record]['fs']
print(f"Sampling frequency for record {first_record}: {fs_first_record} Hz")

## Example of ECG Signal

In [ ]:
# For the first subject, plot all the ECG signals in a 3x4 grid (up to 12 signals)
first_subject = subject_infos_df.iloc[0]
record_path = FOLDER_PATH / first_subject['folder_name'] / first_subject['record_name']
record = wfdb.rdrecord(str(record_path))
num_signals = record.p_signal.shape[1]

fig, axs = plt.subplots(4, 3, figsize=(18, 20), dpi = 300)  # 4 rows, 3 columns

for i in range(12):
    row = i // 3
    col = i % 3
    ax = axs[row, col]
    if i < num_signals:
        ax.plot(record.p_signal[:, i])
        ax.set_title(f'Signal {i+1}')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

## Time Series Understanding

Given a time series, it is important to understand its underlying behavior. For ECG signals, we are particularly interested in identifying:

- **Trends**
- **Seasonality**
- **Cyclicity**

Unlike many other types of time series, ECG signals inherently exhibit these characteristics due to their physiological nature. Therefore, we do not need to apply specialized algorithms to detect trends, seasonality, or cyclicity—they are present by definition in the ECG data.

### Noise removal

In this section, we address the challenge of noise in ECG time series data. We present and compare several techniques for denoising, including rolling window smoothing, low-pass, high-pass, and band-pass filtering. These methods help to enhance the quality of the signals by reducing artifacts and unwanted fluctuations, ensuring that subsequent analyses and feature extraction are based on clean and reliable data.

#### Rolling window approach

The first idea to remove noise in the time series is to use a rolling window (with a defeault value of 5), to remove the noise from the time series. The method operates by sliding a fixed-size window of length $w$ sequentially across the time series. For each time step $t$, the algorithm isolates the subset of data points falling within the window $[t-w+1, t]$ and computes a summary statistic—typically the arithmetic mean or median—to replace the original value $x_t$. 

In [ ]:
# Rolling window smoothing and variability estimation for time series denoising

def moving_average(signal, window_size=5):
    """
    Apply a moving average filter to a 1D numpy array signal.

    This function smooths the input signal by replacing each value with the mean of its neighbors
    within a window of specified size, centered at each point. The result is a signal with reduced
    high-frequency noise, making underlying trends and patterns more visible. The output is standardized
    (zero mean, unit variance) before smoothing, and the result is a pandas Series with NaN values at the edges
    where the window cannot be fully applied.

    Parameters
    ----------
    signal : numpy.ndarray
        Input 1D signal to be smoothed.
    window_size : int, optional
        Size of the moving window (default is 5).

    Returns
    -------
    pandas.Series
        Smoothed signal, with NaN at the edges.
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)

    # Apply centered rolling window average
    filtered = signal_series.rolling(window=window_size).mean()

    return filtered

def moving_std(signal, window_size=5):
    """
    Apply a moving standard deviation filter to a 1D numpy array signal.

    This function estimates the local variability of the signal by computing the standard deviation
    within a centered window of specified size at each point. The signal is first standardized, and
    the result is a pandas Series with NaN values at the edges where the window cannot be fully applied.

    Parameters
    ----------
    signal : numpy.ndarray
        Input 1D signal.
    window_size : int, optional
        Size of the moving window (default is 5).

    Returns
    -------
    pandas.Series
        Local standard deviation (of standardized signal), with NaN at the edges.
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)

    # Standardize: (x - mean) / std
    standardized = (signal_series - signal_series.mean()) / signal_series.std()

    # Apply centered rolling window standard deviation
    filtered = standardized.rolling(window=window_size, center=True).std()

    return filtered

#### Low/High-Pass and Band-Pass filters

An alternative to time-domain smoothing is the application of frequency-selective filters. These filters are essential for isolating the physiological signal from noise sources that occupy distinct frequency bands.
- **Low-Pass Filter (LPF)**:An LPF is designed to transmit signals with frequencies below a specified cutoff frequency ($f_c$) while attenuating frequencies above this threshold. In ECG processing, LPFs are critical for removing high-frequency artifacts, such as electromyographic (EMG) noise from muscle tremors or 50/60 Hz power line interference.
- **High-Pass Filter (HPF)**:Conversely, an HPF permits signals with frequencies above a selected cutoff to pass, suppressing lower frequencies. This is primarily used to eliminate low-frequency noise, such as baseline wander caused by respiration or patient movement, which can distort the ST-segment and affect diagnostic accuracy.
- **Band-Pass Filter (BPF)**:A Band-Pass filter combines the properties of both LPF and HPF to isolate a specific range of frequencies known as the passband. It rejects signal components both below a lower cutoff and above an upper cutoff. This is often the preferred method for QRS detection, as it focuses on the specific frequency range (typically 5–15 Hz) where the energy of the QRS complex is concentrated, effectively filtering out both baseline drift and high-frequency noise simultaneously.

In [ ]:
def low_pass_filter(signal, fs, cutoff=30, order=5):
    """
    Apply a low-pass Butterworth filter to the input signal.

    This filter removes high-frequency noise above the specified cutoff frequency,
    such as muscle artifacts or power line interference, while preserving the lower-frequency
    components of the ECG (e.g., P, QRS, and T waves). It is commonly used to smooth the signal
    and enhance the main cardiac features.

    Parameters
    ----------
    signal : numpy.ndarray
        Input 1D ECG signal.
    fs : float
        Sampling frequency of the signal (Hz).
    cutoff : float, optional
        Cutoff frequency for the low-pass filter (Hz). Default is 30 Hz.
    order : int, optional
        Order of the Butterworth filter. Higher order means a steeper roll-off. Default is 5.

    Returns
    -------
    numpy.ndarray
        Filtered signal with high-frequency noise attenuated.
    """
    # Calculate the Nyquist frequency (half the sampling rate)
    nyquist = 0.5 * fs

    # Normalize the cutoff frequency with respect to Nyquist
    normal_cutoff = cutoff / nyquist

    # Design Butterworth filter coefficients
    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    # Apply the filter to the signal using zero-phase filtering
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def high_pass_filter(signal, fs, cutoff=0.5, order=5):
    """
    Apply a high-pass Butterworth filter to the input signal.

    This filter removes low-frequency noise below the specified cutoff frequency,
    such as baseline wander caused by respiration or patient movement, while preserving
    the higher-frequency components of the ECG. It is useful for stabilizing the baseline
    and improving the accuracy of feature detection.

    Parameters
    ----------
    signal : numpy.ndarray
        Input 1D ECG signal.
    fs : float
        Sampling frequency of the signal (Hz).
    cutoff : float, optional
        Cutoff frequency for the high-pass filter (Hz). Default is 0.5 Hz.
    order : int, optional
        Order of the Butterworth filter. Default is 5.

    Returns
    -------
    numpy.ndarray
        Filtered signal with low-frequency drift removed.
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def band_pass_filter(signal, fs, lowcut=0.5, highcut=30, order=5):
    """
    Apply a band-pass Butterworth filter to the input signal.

    This filter allows only the frequencies within the specified range (passband) to remain,
    attenuating both low-frequency (e.g., baseline wander) and high-frequency (e.g., muscle noise)
    components. Band-pass filtering is especially effective for isolating the QRS complex,
    which is concentrated in a specific frequency band.

    Parameters
    ----------
    signal : numpy.ndarray
        Input 1D ECG signal.
    fs : float
        Sampling frequency of the signal (Hz).
    lowcut : float, optional
        Lower cutoff frequency of the passband (Hz). Default is 0.5 Hz.
    highcut : float, optional
        Upper cutoff frequency of the passband (Hz). Default is 30 Hz.
    order : int, optional
        Order of the Butterworth filter. Default is 5.

    Returns
    -------
    numpy.ndarray
        Filtered signal containing only the desired frequency band.
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

## Time series data quality

In [ ]:
# For each subject, take leads V5 
subj_dict = {}
for subject_id, row in subject_infos_df.iterrows():
    record_name = row['record_name']
    data = timeseries_dict[record_name]
    signal = data['signal']
    fs = data['fs']
    num_samples = data['samples_per_channel']
    
    # Extract lead V5
    lead_V5 = signal[4 * num_samples : 5 * num_samples]

    # Shift based on the rolling mean to remove baseline wander
    lead_V5 = lead_V5 - moving_average(lead_V5, window_size=int(fs*0.2)).fillna(0).values
    
    subject_id = subject_infos_df.iloc[subject_id]['subject_id']
    subj_dict[subject_id] = {
        'V5': lead_V5,
        'fs': fs
    }

In [ ]:
# Initialize a standard scaler for normalization
scaler = StandardScaler()

# Select the first subject's ID from the subject dictionary
subject_id = list(subj_dict.keys())[0]

# Retrieve the original (pre-scaled) V5 lead signal for the selected subject
original_V5_signal = subj_dict[subject_id]['V5']

# Get the sampling frequency for the selected subject
fs = subj_dict[subject_id]['fs']

# Standardize the V5 signal (zero mean, unit variance) for further processing
scaled_V5_signal = scaler.fit_transform(original_V5_signal.reshape(-1, 1)).flatten()

#### Noise cancellation approach 1: Using the rolling window to clean the time series

In [ ]:
mavg_filtered_V5 = moving_average(scaled_V5_signal, window_size=5).fillna(0).values

#### Noise cancellation approach 2: Using the Band-Pass filter to cancel the noise in the time series

In [ ]:
bp_filtered_V5 = band_pass_filter(scaled_V5_signal, fs, lowcut=0.5, highcut=30, order=5)

#### Comparison of the two methods

In [ ]:
# Calculate the differences between the filtered and the original (Lead V5)
diff_ma_V5 = mavg_filtered_V5 - scaled_V5_signal
diff_bp_V5 = bp_filtered_V5 - scaled_V5_signal

print(f"Lead V5 - Difference MA Filter - Mean: {np.mean(diff_ma_V5):.4f}, Std: {np.std(diff_ma_V5):.4f}")
print(f"Lead V5 - Difference BP Filter - Mean: {np.mean(diff_bp_V5):.4f}, Std: {np.std(diff_bp_V5):.4f}")

# Plot Lead V5: original, moving average filtered, and band-pass filtered
fig, axes = plt.subplots(3, 1, figsize=(12, 10), dpi=300)

# Plot 1: Original Signal - Lead V5
axes[0].plot(scaled_V5_signal, label='Original Signal', color='blue', alpha=0.7)
axes[0].set_title(f'Original Signal (Lead V5) - Subject {subject_id}')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Plot 2: Moving Average Filtered - Lead V5
axes[1].plot(mavg_filtered_V5, label='Moving Average Filtered', color='red', alpha=0.7)
axes[1].set_title(f'Moving Average Filtered (Lead V5) - Subject {subject_id}')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

# Plot 3: Band-Pass Filtered - Lead V5
axes[2].plot(bp_filtered_V5, label='Band-Pass Filtered', color='green', alpha=0.7)
axes[2].set_title(f'Band-Pass Filtered (Lead V5) - Subject {subject_id}')
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
handles = [axes[0].get_lines()[0], axes[1].get_lines()[0], axes[2].get_lines()[0]]
labels = ['Original Signal', 'Moving Average Filtered', 'Band-Pass Filtered']
# Place the legend at the top center of the figure
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=True)
plt.show()

In [ ]:
# For all the subjects, use the band-pass filter and save the filtered signals
for subject_id, data in subj_dict.items():
    fs = data['fs']
    lead_V5 = data['V5']
    
    # Scale the signals
    scaled_V5_signal = scaler.fit_transform(lead_V5.reshape(-1, 1)).flatten()
    
    # Apply band-pass filter
    bp_filtered_V5 = band_pass_filter(scaled_V5_signal, fs, lowcut=0.5, highcut=30, order=5)
    
    # Update the dictionary with filtered signals
    subj_dict[subject_id]['bp_filtered_V5'] = bp_filtered_V5

#### Savitzky-Golay smoothing
The Savitzky-Golay filter is a digital signal processing technique used to smooth data and increase the signal-to-noise ratio (SNR) while preserving the high-frequency components of the signal, such as peak heights and widths.
Unlike standard moving average filters, which simply calculate the mean of a data window (often leading to signal distortion or "flattening" of sharp peaks), the Savitzky-Golay method performs a local polynomial regression.
1) For each point $x_t$ in the time series, the algorithm:Selects a symmetric window of $2m+1$ data points centered at $x_t$.
2) Fits a polynomial of order $p$ (typically $p=2$ or $p=3$) to these points using the method of linear least squares.
3) Replaces the original value $x_t$ with the value of the fitted polynomial at that central position.

In [ ]:
from scipy.signal import savgol_filter

def smooth_cardiac_signal(signal, window_size=15, poly_order=3):
    """
    Apply Savitzky-Golay smoothing to an ECG signal to reduce noise while preserving sharp features like R-peaks.

    Parameters
    ----------
    signal : array-like
        The 1D ECG signal to be smoothed.
    window_size : int, optional
        The length of the filter window (number of coefficients). Must be odd.
        For ECG sampled at 500Hz, a value between 15 and 31 is usually effective.
    poly_order : int, optional
        The order of the polynomial used to fit the samples. 2 or 3 is typical for cardiac data.

    Returns
    -------
    smoothed_signal : ndarray
        The filtered signal, with noise reduced but R-peak heights preserved.

    Notes
    -----
    The Savitzky-Golay filter is particularly useful for ECG signals because it smooths the signal
    without distorting the amplitude of sharp features (such as QRS complexes), unlike simple moving averages.
    """
    # Apply the Savitzky-Golay filter to the input signal
    return savgol_filter(signal, window_size, poly_order)

In [ ]:
# Take the bp-filtered signal for the first subject
subject_id = list(subj_dict.keys())[0]
bp_filtered_V5 = subj_dict[subject_id]['bp_filtered_V5']

# Apply Savitzky-Golay smoothing for the first subject
smoothed_V5 = smooth_cardiac_signal(bp_filtered_V5, window_size=15, poly_order=3)

# Plot Lead V5: Moving Average Filtered and Savitzky-Golay Smoothed side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=300)

# Left: Moving Average Filtered
axes[0].plot(mavg_filtered_V5, label='Moving Average Filtered', color='red', alpha=0.7)
axes[0].set_title(f'Moving Average Filtered (Lead V5) - Subject {subject_id}')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper right')

# Right: Savitzky-Golay Smoothed
axes[1].plot(smoothed_V5, label='Savitzky-Golay Smoothed', color='purple', alpha=0.7)
axes[1].set_title(f'Savitzky-Golay Smoothed (Lead V5) - Subject {subject_id}')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

# Print the difference between the smoothed and moving average filtered signals for Lead V5
diff_V5 = smoothed_V5 - mavg_filtered_V5
print(f"Lead V5 - Difference (Savitzky-Golay - Moving Average) - Mean: {np.mean(diff_V5):.4f}, Std: {np.std(diff_V5):.4f}")

In [ ]:
# For each subject, apply the savitzky-golay filter and save the smoothed signals
for subject_id, data in subj_dict.items():
    bp_filtered_V5 = data['bp_filtered_V5']
    
    # Apply Savitzky-Golay smoothing
    smoothed_V5 = smooth_cardiac_signal(bp_filtered_V5, window_size=15, poly_order=3)
    
    # Update the dictionary with smoothed signals
    subj_dict[subject_id]['smoothed_V5'] = smoothed_V5

### Feature Extraction: NeuroKit2 and Custom Methods

In this section, we extract a comprehensive set of features from the ECG signals to enable robust downstream analysis such as classification or clustering. We use both the NeuroKit2 library and custom feature extraction methods to capture a wide range of physiological and morphological information from each subject's ECG.

**Features extracted using NeuroKit2:**
- **Heart Rate (bpm):** Average heart rate computed from detected R-peaks.
- **HRV Time-Domain Metrics:**
  - **SDNN:** Standard deviation of NN (normal-to-normal) intervals, reflecting overall heart rate variability.
  - **RMSSD:** Root mean square of successive differences between NN intervals, sensitive to short-term HRV.
  - **pNN50:** Percentage of successive NN intervals differing by more than 50 ms.
- **Morphological Template Features:**
  - **R-peak Amplitude:** Maximum amplitude of the ensemble-averaged heartbeat template.
  - **QRS Duration:** Estimated width of the QRS complex at half-max amplitude (in ms).
  - **Template Kurtosis:** Kurtosis of the average heartbeat, indicating the "peakedness" of the QRS complex.
- **Ischemia-Related Features:**
  - **Mean T-Wave Amplitude:** Average amplitude difference between T-peak and baseline.
  - **Mean ST Deviation:** Average deviation at the J-point (end of QRS) relative to baseline.
  - **Mean QT Interval:** Average duration from Q onset to T offset (in seconds).
  - **Mean QTc Bazett:** QT interval corrected for heart rate using Bazett's formula.

These features collectively capture both the temporal dynamics and morphological characteristics of the ECG, providing a rich representation for subsequent machine learning or statistical analysis.

In [ ]:
from scipy import stats
import neurokit2 as nk
def extract_ecg_features_neurokit(signal, fs, window_ms=600):
    """
    Extract a comprehensive set of ECG features using NeuroKit2, including HRV, heart rate, morphology, and ischemia markers.

    Parameters
    ----------
    signal : array-like
        The 1D ECG signal to analyze.
    fs : float
        Sampling frequency of the signal (Hz).
    window_ms : int, optional
        Window size in milliseconds for heartbeat segmentation and template extraction (default: 600 ms).

    Returns
    -------
    dict
        Dictionary containing HRV metrics, heart rate, morphological features, and ischemia-related features.
        Returns None if the signal is too short or R-peaks cannot be reliably detected.

    Notes
    -----
    This function leverages NeuroKit2 for robust ECG processing and feature extraction, suitable for downstream
    machine learning or clinical analysis.
    """
    # Signal Preprocessing
    # Clean the raw ECG signal using the Hamilton 2002 method to remove noise and baseline wander
    signal_clean = nk.ecg_clean(signal, sampling_rate=fs, method="hamilton2002")

    # Detect R-peaks in the cleaned ECG signal
    _, info = nk.ecg_peaks(signal_clean, sampling_rate=fs)
    r_peaks = info["ECG_R_Peaks"]

    # If fewer than 2 R-peaks are detected, return None (insufficient data for feature extraction)
    if len(r_peaks) < 2:
        return None

    # Heart Rate Variability (HRV) and Heart Rate 
    # Compute time-domain HRV metrics using NeuroKit2
    hrv_df = nk.hrv_time(info, sampling_rate=fs, show=False)

    # Calculate mean NN interval and heart rate (bpm)
    mean_nn = hrv_df['HRV_MeanNN'].values[0]
    heart_rate = 60000 / mean_nn if mean_nn != 0 else 0

    # Collect HRV features in a dictionary
    hrv_features = {
        'Heart_Rate': heart_rate,
        'SDNN': hrv_df['HRV_SDNN'].values[0],
        'RMSSD': hrv_df['HRV_RMSSD'].values[0],
        'pNN50': hrv_df['HRV_pNN50'].values[0]
    }

    # Morphological Template Extraction
    # Segment the ECG signal into individual heartbeats centered on R-peaks
    window_sec = window_ms / 1000
    epochs = nk.ecg_segment(signal_clean, rpeaks=r_peaks, sampling_rate=fs,
                            show=False, tmin=-window_sec/2, tmax=window_sec/2)

    # Compute the ensemble average (template) of all segmented heartbeats
    df_epochs = pd.concat([epochs[i]['Signal'] for i in epochs], axis=1)
    template = df_epochs.mean(axis=1).values

    # Extract morphological features from the template
    r_amplitude = np.max(template)  # Maximum amplitude (R-peak)
    template_kurtosis = stats.kurtosis(template)  # Kurtosis (peakiness)
    # QRS duration estimated as the width at half-max amplitude
    threshold = r_amplitude * 0.5
    above_threshold_indices = np.where(template > threshold)[0]
    qrs_duration_ms = ((above_threshold_indices[-1] - above_threshold_indices[0]) / fs * 1000) if len(above_threshold_indices) > 0 else np.nan

    morphology_features = {
        "template": template,
        "r_amplitude": r_amplitude,
        "qrs_duration_ms": qrs_duration_ms,
        "template_kurtosis": template_kurtosis
    }

    # Ischemia-Related Features (Wave-Specific)
    try:
        # Delineate ECG waves (Q, T, S, etc.) using discrete wavelet transform
        _, waves = nk.ecg_delineate(signal_clean, r_peaks, sampling_rate=fs, method="dwt")

        # Helper function to safely extract wave indices or return NaNs if missing
        def get_wave(waves, key, n):
            return np.array(waves[key]) if key in waves else np.full(n, np.nan)

        n_beats = len(r_peaks)
        idx_q_onset = get_wave(waves, "ECG_Q_Peaks", n_beats)
        idx_t_peak = get_wave(waves, "ECG_T_Peaks", n_beats)
        idx_t_offset = get_wave(waves, "ECG_T_Offsets", n_beats)
        idx_s_offset = get_wave(waves, "ECG_S_Offsets", n_beats)

        # Helper to extract amplitude at given indices, handling out-of-bounds and NaNs
        def get_amplitude(signal, indices):
            indices = np.array(indices)
            values = np.full(indices.shape, np.nan)
            valid = ~np.isnan(indices) & (indices < len(signal)) & (indices >= 0)
            values[valid] = signal[indices[valid].astype(int)]
            return values

        # Calculate baseline, T-peak, and J-point amplitudes
        baseline_amp = get_amplitude(signal_clean, idx_q_onset)
        t_peak_amp = get_amplitude(signal_clean, idx_t_peak)
        j_point_amp = get_amplitude(signal_clean, idx_s_offset)

        # Compute ischemia-related features
        t_wave_amplitude = t_peak_amp - baseline_amp
        st_deviation = j_point_amp - baseline_amp
        qt_interval_sec = (idx_t_offset - idx_q_onset) / fs

        # RR intervals and QTc Bazett calculation
        rr_intervals_samples = np.diff(r_peaks, prepend=np.nan)
        rr_intervals_sec = rr_intervals_samples / fs
        qtc_bazett = qt_interval_sec / np.sqrt(rr_intervals_sec)

        # Aggregate results in a DataFrame for easy nanmean calculation
        results = pd.DataFrame({
            "R_Peak_Index": r_peaks,
            "T_Wave_Amplitude": t_wave_amplitude,
            "ST_Deviation": st_deviation,
            "QT_Interval_Sec": qt_interval_sec,
            "RR_Interval_Sec": rr_intervals_sec,
            "QTc_Bazett": qtc_bazett
        })

        # Helper to compute nanmean safely
        def safe_nanmean(arr):
            arr = np.asarray(arr)
            if arr.size == 0 or np.all(np.isnan(arr)):
                return np.nan
            return np.nanmean(arr)

        # Collect ischemia features as means across beats
        ischemia_features = {
            "mean_t_wave_amplitude": safe_nanmean(results["T_Wave_Amplitude"].values),
            "mean_st_deviation": safe_nanmean(results["ST_Deviation"].values),
            "mean_qt_interval_sec": safe_nanmean(results["QT_Interval_Sec"].values),
            "mean_qtc_bazett": safe_nanmean(results["QTc_Bazett"].values)
        }
    except Exception as e:
        # If delineation fails, fill ischemia features with NaN
        print(f"Ischemia feature extraction failed: {e}")
        ischemia_features = {
            "mean_t_wave_amplitude": np.nan,
            "mean_st_deviation": np.nan,
            "mean_qt_interval_sec": np.nan,
            "mean_qtc_bazett": np.nan
        }

    # Combine all features into a single dictionary
    all_features = {}
    all_features.update(hrv_features)
    all_features.update(morphology_features)
    all_features.update(ischemia_features)
    return all_features

In [ ]:
import pandas as pd

# Assuming you have a dictionary or list of your data
# data_structure = {'Subj_01': signal_array_1, 'Subj_02': signal_array_2, ...}

scalar_results = []   # For SDNN, RMSSD, Kurtosis, etc.
templates_dict = {}   # Store templates separately (as they are arrays)

# 1. Loop through subjects
for subj_id, data in subj_dict.items():
    signal = data['smoothed_V5']
    fs = data['fs']
    
    # 2. Call the function
    # Note: Ensure 'fs' matches your specific device (e.g., 250, 500, 1000)
    features = extract_ecg_features_neurokit(signal, fs=fs, window_ms=600)
    
    # 3. Check for Valid Output (Handle None)
    if features is not None:
        # Separate the Template (Array) from the Scalars (Numbers)
        # We pop it out so it doesn't mess up the pandas DataFrame later
        template_array = features.pop('template')
        
        # Store the template separately
        templates_dict[subj_id] = template_array
        
        # Add the ID to the scalar features for tracking
        features['subject_id'] = subj_id
        features['smoothed_V5'] = signal 
        scalar_results.append(features)
        
    else:
        print(f"Skipping {subj_id}: Signal too short or noisy.")

# 4. Create your Final DataFrame for Classification
features_df = pd.DataFrame(scalar_results)

#Make sure Subject_ID is the first column
cols = features_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('subject_id')))
features_df = features_df[cols]

# Drop st_deviation column due to high missingness
features_df = features_df.drop(columns=['mean_st_deviation'])

# Drop the rows with mean_qt_interval_sec missing values and mean_qtc_bazett missing values
features_df = features_df.dropna(subset=['mean_qt_interval_sec', 'mean_qtc_bazett'])

features_df

After creating the new DataFrame with the features, we merge it with the heart diagnoses DataFrame so that we only keep patients for whom we have clinical data.

In [ ]:
# Load the heart disease labels or metadata for each subject
hd_df = pd.read_csv('../../data/processed/hd.csv')

# Ensure the subject_id column is of integer type for a reliable merge
features_df['subject_id'] = features_df['subject_id'].astype(int)

# Merge extracted ECG features with heart disease labels on subject_id (inner join keeps only matched subjects)
full_df = pd.merge(features_df, hd_df, on='subject_id', how='inner')

# Save the merged DataFrame as a pickle file to preserve any numpy arrays or complex objects
full_df.to_pickle('../../data/processed/time_series_features.pkl')

# Display the final merged DataFrame
full_df